# GRU — ablaciones: aporte de la secuencia y dependencia del orden del CSV

**Objetivo:** medir en **validación** si

- **(a) la secuencia aporta:** W=1 (solo el flujo objetivo) contra W=10 (baseline);
- **(b) el modelo depende del orden del CSV dentro de cada minuto:** variante `aleatorio` (mismas filas por split, orden sorteado dentro de cada minuto) contra la variante `csv` del baseline.

Test no se usa en este notebook.

Misma `CONFIG` y mismas `SEMILLAS` que `01_baseline.ipynb`: units 32, dropout 0.3, learning rate 1e-3, batch 64, epochs 50, patience 10; semillas 0–4.

## Reglas de decisión (fijadas antes de correr)

Se compara la media de cada ablación contra la media y la desviación estándar del baseline (`results/gru/01_baseline/baseline_w10_seeds.json`):

- **"La secuencia aporta"** si la media de `recall_05` de W=1 < media_base − 2·std_base, **o** la media de `pr_auc` de W=1 < media_base − 2·std_base.
- **"Hay dependencia del orden del CSV"** con el mismo criterio, aplicado a la variante `aleatorio`.
- Si ninguna condición se cumple: **"sin diferencia detectable en validación"**.

## 1. Imports y configuración

In [1]:
import sys
sys.path.append("../..")  # raíz del repo, para poder importar src.*

import gc
import json
import os

import numpy as np
import pandas as pd

from src.models.gru.experiments import correr_experimento

In [2]:
CONFIG = {
    "units": 32,
    "dropout": 0.3,
    "learning_rate": 1e-3,
    "batch_size": 64,
    "epochs": 50,
    "patience": 10,
}
SEMILLAS = [0, 1, 2, 3, 4]

W = 10
DATA_DIR = "../../data/sequences"
MODELS_DIR = "../../models/gru/02_ablaciones"
RESULTS_DIR = "../../results/gru/02_ablaciones"
BASELINE_PATH = "../../results/gru/01_baseline/baseline_w10_seeds.json"
ATAQUES_VAL_BASELINE = 1884

with open(BASELINE_PATH, encoding="utf-8") as f:
    baseline = json.load(f)

# Las ablaciones solo son comparables si comparten configuración y semillas con el baseline
assert baseline["semillas"] == SEMILLAS, "SEMILLAS distintas a las del baseline"
assert all(baseline["config"].get(k) == v for k, v in CONFIG.items()), "CONFIG distinta a la del baseline"
print("Baseline:", baseline["modelo"], "| config:", baseline["config"])

Baseline: baseline_w10 | config: {'units': 32, 'dropout': 0.3, 'learning_rate': 0.001, 'batch_size': 64, 'epochs': 50, 'patience': 10}


## 2. Ablación W=1

Se toma el último paso de cada ventana W=10 (variante csv): la misma fila objetivo y la misma etiqueta que en el baseline, sin los 9 flujos anteriores.

In [3]:
X_train_w10 = np.load(f"{DATA_DIR}/X_train_w{W}.npy")
y_train = np.load(f"{DATA_DIR}/y_train_w{W}.npy")
X_val_w10 = np.load(f"{DATA_DIR}/X_val_w{W}.npy")
y_val = np.load(f"{DATA_DIR}/y_val_w{W}.npy")

# .copy(): sin él, el recorte es una vista y X_*_w10 seguiría ocupando RAM tras el del
X_train = X_train_w10[:, -1:, :].copy()
X_val = X_val_w10[:, -1:, :].copy()

# Las etiquetas son las de W=10 sin modificar: una por ventana, la de su último paso
assert X_train.shape == (len(y_train), 1, 59), X_train.shape
assert X_val.shape == (len(y_val), 1, 59), X_val.shape
assert len(y_train) == len(X_train_w10) and len(y_val) == len(X_val_w10)
assert np.array_equal(X_val[:, 0, :], X_val_w10[:, -1, :])
assert int(y_val.sum()) == ATAQUES_VAL_BASELINE, int(y_val.sum())

del X_train_w10, X_val_w10
gc.collect()

print(f"X_train: {X_train.shape} | ataques: {int(y_train.sum())}")
print(f"X_val:   {X_val.shape} | ataques: {int(y_val.sum())}")

X_train: (310484, 1, 59) | ataques: 8230
X_val:   (66496, 1, 59) | ataques: 1884


In [4]:
tabla_w1 = correr_experimento(
    "w1", CONFIG, X_train, y_train, X_val, y_val, SEMILLAS, MODELS_DIR, RESULTS_DIR,
)


w1 — semilla 0

Epoch 1/50
4852/4852 ━━━━━━━━━━━━━━━━━━━━ 5s 907us/step - accuracy: 0.9877 - auc: 0.9986 - loss: 0.0358 - precision: 0.6850 - recall: 0.9937 - val_accuracy: 0.9954 - val_auc: 0.9991 - val_loss: 0.0191 - val_precision: 0.8619 - val_recall: 0.9973
Epoch 2/50
4852/4852 ━━━━━━━━━━━━━━━━━━━━ 4s 871us/step - accuracy: 0.9970 - auc: 0.9993 - loss: 0.0130 - precision: 0.8992 - recall: 0.9977 - val_accuracy: 0.9956 - val_auc: 0.9991 - val_loss: 0.0165 - val_precision: 0.8683 - val_recall: 0.9973
Epoch 3/50
4852/4852 ━━━━━━━━━━━━━━━━━━━━ 4s 883us/step - accuracy: 0.9974 - auc: 0.9992 - loss: 0.0118 - precision: 0.9126 - recall: 0.9978 - val_accuracy: 0.9954 - val_auc: 0.9991 - val_loss: 0.0140 - val_precision: 0.8763 - val_recall: 0.9740
Epoch 4/50
4852/4852 ━━━━━━━━━━━━━━━━━━━━ 5s 934us/step - accuracy: 0.9977 - auc: 0.9993 - loss: 0.0109 - precision: 0.9235 - recall: 0.9974 - val_accuracy: 0.9953 - val_auc: 0.9991 - val_loss: 0.0144 - val_precision: 0.8731 - val_recall: 0.9751

In [5]:
tabla_w1

,val_loss,mejor_epoca,epocas_entrenadas,tiempo_entrenamiento_s,roc_auc,pr_auc,recall_05,precision_05,umbral_f2,f2,recall_f2,precision_f2,fn_f2,fp_f2
0,0.006758,36.000000,46.000000,217.483526,0.999403,0.991779,0.968684,0.984358,0.010000,0.972208,0.998938,0.878208,2.000000,261.000000
1,0.003528,25.000000,35.000000,177.536979,0.999727,0.994711,0.998408,0.985333,0.472000,0.995765,0.998408,0.985333,3.000000,28.000000
2,0.005068,19.000000,29.000000,137.596030,0.999783,0.994153,0.969214,0.984898,0.113000,0.987816,0.998408,0.947607,3.000000,104.000000
3,0.007723,4.000000,14.000000,70.757600,0.999600,0.992315,0.997877,0.885539,0.507000,0.976420,0.997877,0.899091,4.000000,211.000000
4,0.006637,14.000000,24.000000,115.554947,0.999744,0.992627,0.968153,0.980118,0.999000,0.972267,0.967622,0.991300,61.000000,16.000000
media,0.005943,19.600000,29.600000,143.785816,0.999651,0.993117,0.980467,0.964049,0.420200,0.980895,0.992251,0.940308,14.600000,124.000000
std,0.001652,11.970798,11.970798,56.433388,0.000155,0.001254,0.016141,0.043938,0.389828,0.010476,0.013773,0.050586,25.948025,109.107745


In [6]:
# Liberar antes de cargar la siguiente ablación, para no tener ambos sets en RAM
del X_train, y_train, X_val, y_val
gc.collect()

41502

## 3. Ablación aleatorio (W=10)

Secuencias `_aleatorio` de `scripts/gru/build_gru_sequences.py`: mismas filas en cada split y mismo scaler que la variante csv; solo cambia el orden de las filas dentro de cada (bloque, Timestamp).

In [7]:
X_train = np.load(f"{DATA_DIR}/X_train_w{W}_aleatorio.npy")
y_train = np.load(f"{DATA_DIR}/y_train_w{W}_aleatorio.npy")
X_val = np.load(f"{DATA_DIR}/X_val_w{W}_aleatorio.npy")
y_val = np.load(f"{DATA_DIR}/y_val_w{W}_aleatorio.npy")

assert X_train.shape[1:] == X_val.shape[1:] == (W, 59)
assert int(y_val.sum()) == ATAQUES_VAL_BASELINE, (
    f"y_val tiene {int(y_val.sum())} ataques; el baseline tiene {ATAQUES_VAL_BASELINE}"
)

print(f"X_train: {X_train.shape} | ataques: {int(y_train.sum())}")
print(f"X_val:   {X_val.shape} | ataques: {int(y_val.sum())}")

X_train: (310484, 10, 59) | ataques: 8224
X_val:   (66496, 10, 59) | ataques: 1884


In [8]:
tabla_aleatorio = correr_experimento(
    "aleatorio_w10", CONFIG, X_train, y_train, X_val, y_val, SEMILLAS, MODELS_DIR, RESULTS_DIR,
)


aleatorio_w10 — semilla 0
Epoch 1/50
4852/4852 ━━━━━━━━━━━━━━━━━━━━ 12s 2ms/step - accuracy: 0.9860 - auc: 0.9987 - loss: 0.0394 - precision: 0.6565 - recall: 0.9906 - val_accuracy: 0.9966 - val_auc: 0.9995 - val_loss: 0.0102 - val_precision: 0.8962 - val_recall: 0.9942
Epoch 2/50
4852/4852 ━━━━━━━━━━━━━━━━━━━━ 10s 2ms/step - accuracy: 0.9960 - auc: 0.9994 - loss: 0.0133 - precision: 0.8718 - recall: 0.9974 - val_accuracy: 0.9958 - val_auc: 0.9991 - val_loss: 0.0119 - val_precision: 0.8754 - val_recall: 0.9920
Epoch 3/50
4852/4852 ━━━━━━━━━━━━━━━━━━━━ 10s 2ms/step - accuracy: 0.9968 - auc: 0.9995 - loss: 0.0105 - precision: 0.8918 - recall: 0.9985 - val_accuracy: 0.9961 - val_auc: 0.9992 - val_loss: 0.0104 - val_precision: 0.8947 - val_recall: 0.9788
Epoch 4/50
4852/4852 ━━━━━━━━━━━━━━━━━━━━ 10s 2ms/step - accuracy: 0.9971 - auc: 0.9996 - loss: 0.0094 - precision: 0.9025 - recall: 0.9983 - val_accuracy: 0.9976 - val_auc: 0.9992 - val_loss: 0.0075 - val_precision: 0.9417 - val_recall: 

In [9]:
tabla_aleatorio

,val_loss,mejor_epoca,epocas_entrenadas,tiempo_entrenamiento_s,roc_auc,pr_auc,recall_05,precision_05,umbral_f2,f2,recall_f2,precision_f2,fn_f2,fp_f2
0,0.006295,15.000000,25.000000,253.960603,0.999731,0.994476,0.973992,0.969873,0.106000,0.974558,0.984076,0.938259,30.000000,122.000000
1,0.004722,13.000000,23.000000,219.800637,0.999899,0.998261,0.985669,0.968701,0.125000,0.988122,0.997877,0.950936,4.000000,97.000000
2,0.005228,19.000000,29.000000,285.250925,0.999906,0.997150,0.973461,0.972944,0.066000,0.975623,0.985669,0.937405,27.000000,124.000000
3,0.005860,13.000000,23.000000,216.104817,0.999816,0.996090,0.970807,0.965172,0.070000,0.978101,0.990977,0.929781,17.000000,141.000000
4,0.005060,34.000000,44.000000,353.595832,0.999826,0.996817,0.975584,0.975066,0.103000,0.978307,0.986200,0.947959,26.000000,102.000000
media,0.005433,18.800000,28.800000,265.742563,0.999836,0.996559,0.975902,0.970351,0.094000,0.978942,0.988960,0.940868,20.800000,117.200000
std,0.000635,8.843076,8.843076,56.595393,0.000072,0.001403,0.005724,0.003832,0.025229,0.005376,0.005610,0.008564,10.568822,17.852171


In [10]:
del X_train, y_train, X_val, y_val
gc.collect()

40918

## 4. Tabla comparativa (validación)

Media ± desviación estándar muestral sobre las 5 semillas, y diferencia de medias contra el baseline (Δ = ablación − baseline). Se lee de los `*_seeds.json`, así que esta sección puede volver a ejecutarse sin reentrenar.

In [11]:
METRICAS = ["val_loss", "recall_05", "precision_05", "pr_auc", "f2", "fn_f2", "fp_f2"]


def cargar_resumen(path):
    with open(path, encoding="utf-8") as f:
        resultado = json.load(f)
    return resultado["media"], resultado["std"]


resumenes = {
    "baseline_w10": cargar_resumen(BASELINE_PATH),
    "w1": cargar_resumen(f"{RESULTS_DIR}/w1_seeds.json"),
    "aleatorio_w10": cargar_resumen(f"{RESULTS_DIR}/aleatorio_w10_seeds.json"),
}
media_base, std_base = resumenes["baseline_w10"]

comparacion = {}
filas = {}
for nombre, (media, std) in resumenes.items():
    comparacion[nombre] = {
        m: {
            "media": media[m],
            "std": std[m],
            "diferencia_vs_baseline": media[m] - media_base[m],
        }
        for m in METRICAS
    }
    fila = {}
    for m in METRICAS:
        fila[m] = f"{media[m]:.6g} ± {std[m]:.2g}"
        fila[f"Δ {m}"] = media[m] - media_base[m]
    filas[nombre] = fila

tabla_comparativa = pd.DataFrame.from_dict(filas, orient="index")
tabla_comparativa

,val_loss,Δ val_loss,recall_05,Δ recall_05,precision_05,Δ precision_05,pr_auc,Δ pr_auc,f2,Δ f2,fn_f2,Δ fn_f2,fp_f2,Δ fp_f2
baseline_w10,0.00145534 ± 0.00095,0.000000,0.99586 ± 0.0012,0.000000,0.986814 ± 0.018,0.000000,0.999534 ± 0.00026,0.000000,0.996856 ± 0.0014,0.000000,6.8 ± 3.3,0.0,2.4 ± 1.3,0.0
w1,0.00594282 ± 0.0017,0.004487,0.980467 ± 0.016,-0.015393,0.964049 ± 0.044,-0.022765,0.993117 ± 0.0013,-0.006418,0.980895 ± 0.01,-0.015961,14.6 ± 26,7.8,124 ± 1.1e+02,121.6
aleatorio_w10,0.005433 ± 0.00063,0.003978,0.975902 ± 0.0057,-0.019958,0.970351 ± 0.0038,-0.016463,0.996559 ± 0.0014,-0.002976,0.978942 ± 0.0054,-0.017914,20.8 ± 11,14.0,117.2 ± 18,114.8


In [12]:
# Reglas de decisión del markdown inicial
METRICAS_REGLA = ["recall_05", "pr_auc"]
VEREDICTO_SI_CUMPLE = {
    "w1": "la secuencia aporta",
    "aleatorio_w10": "hay dependencia del orden del CSV",
}
SIN_DIFERENCIA = "sin diferencia detectable en validación"

veredictos = {}
for nombre, texto in VEREDICTO_SI_CUMPLE.items():
    media, _ = resumenes[nombre]
    cumplidas = []
    for m in METRICAS_REGLA:
        umbral = media_base[m] - 2 * std_base[m]
        cumple = media[m] < umbral
        print(
            f"{nombre} | {m}: media={media[m]:.6f} | "
            f"umbral (media_base − 2·std_base)={umbral:.6f} | "
            f"{'CUMPLE' if cumple else 'no cumple'}"
        )
        if cumple:
            cumplidas.append(m)
    veredictos[nombre] = {
        "veredicto": texto if cumplidas else SIN_DIFERENCIA,
        "metricas_que_cumplen": cumplidas,
    }
    print(f"-> {nombre}: {veredictos[nombre]['veredicto']}\n")

w1 | recall_05: media=0.980467 | umbral (media_base − 2·std_base)=0.993439 | CUMPLE
w1 | pr_auc: media=0.993117 | umbral (media_base − 2·std_base)=0.999021 | CUMPLE
-> w1: la secuencia aporta

aleatorio_w10 | recall_05: media=0.975902 | umbral (media_base − 2·std_base)=0.993439 | CUMPLE
aleatorio_w10 | pr_auc: media=0.996559 | umbral (media_base − 2·std_base)=0.999021 | CUMPLE
-> aleatorio_w10: hay dependencia del orden del CSV



In [13]:
COMPARACION_PATH = f"{RESULTS_DIR}/comparacion.json"
os.makedirs(RESULTS_DIR, exist_ok=True)

resultado = {
    "conjunto": "validacion",
    "config": CONFIG,
    "semillas": SEMILLAS,
    "regla": "media de recall_05 o de pr_auc de la ablación < media_base − 2·std_base",
    "comparacion": comparacion,
    "veredictos": veredictos,
}

with open(COMPARACION_PATH, "w", encoding="utf-8") as f:
    json.dump(resultado, f, indent=2, ensure_ascii=False)
print(f"Comparación guardada en: {COMPARACION_PATH}")

Comparación guardada en: ../../results/gru/02_ablaciones/comparacion.json


## 5. Verificación del orden del CSV

La ablación `aleatorio` indica que el resultado depende del orden de las filas dentro de cada minuto. Estas dos celdas revisan qué representa ese orden:

- **Retrocesos de tiempo:** si el archivo estuviera escrito en orden cronológico (por inicio o por fin del flujo), al recorrerlo fila por fila el tiempo casi nunca retrocedería más allá de la resolución de un minuto.
- **Agrupamiento por IP:** compara con qué frecuencia dos filas consecutivas del mismo minuto comparten IP, contra lo esperado si el orden fuera aleatorio.

In [14]:
import pandas as pd
from src.preprocessing.clean import cargar_dataset_limpio

df = cargar_dataset_limpio("../../data/processed/wednesday_low_and_slow_clean.csv")
inicio = df["Timestamp"]
fin = inicio + pd.to_timedelta(df["Flow Duration"], unit="us")

def pct_retrocesos(serie, tolerancia_s):
    d = serie.diff().dropna()
    return 100 * (d < -pd.Timedelta(seconds=tolerancia_s)).mean()

for tol in (60, 120):
    print(f"tolerancia {tol:>3}s | retrocesos por INICIO: {pct_retrocesos(inicio, tol):.3f}% | por FIN: {pct_retrocesos(fin, tol):.3f}%")

tolerancia  60s | retrocesos por INICIO: 32.521% | por FIN: 31.637%
tolerancia 120s | retrocesos por INICIO: 30.563% | por FIN: 30.566%


In [15]:
from src.preprocessing.clean import load_raw_csvs, filter_low_and_slow

raw = filter_low_and_slow(load_raw_csvs("../../data/raw"))
raw["minuto"] = raw["Timestamp"].astype(str).str.strip()
mismo_minuto = raw["minuto"] == raw["minuto"].shift()
grupos = raw.groupby("minuto")

def prob_azar(s):
    p = s.value_counts(normalize=True)
    return (p ** 2).sum()

for col in ["Source IP", "Destination IP"]:
    observado = (raw[col] == raw[col].shift())[mismo_minuto].mean()
    esperado = (grupos[col].apply(prob_azar) * grupos.size()).sum() / grupos.size().sum()
    print(f"{col:15} | consecutivos con la misma IP: {100*observado:.1f}% | esperado si fuera al azar: {100*esperado:.1f}%")

AVISO: se encontraron 8 CSVs con Timestamp (parece el dataset completo de la semana). Se usa automáticamente el que corresponde al miércoles: ..\..\data\raw\GeneratedLabelledFlows\TrafficLabelling\Wednesday-workingHours.pcap_ISCX.csv
Cargado: ..\..\data\raw\GeneratedLabelledFlows\TrafficLabelling\Wednesday-workingHours.pcap_ISCX.csv
Etiquetas encontradas en el CSV: ['BENIGN', 'DoS slowloris', 'DoS Slowhttptest', 'DoS Hulk', 'DoS GoldenEye', 'Heartbleed']
Filas antes del filtro: 692703 | después (solo BENIGN + Low&Slow): 451326
Source IP       | consecutivos con la misma IP: 68.6% | esperado si fuera al azar: 21.7%
Destination IP  | consecutivos con la misma IP: 47.1% | esperado si fuera al azar: 15.4%


## Conclusiones


- **W=1 contra el baseline (W=10, orden csv):** sin los 9 flujos anteriores, el recall con umbral 0.5 baja de 0.9959 ± 0.0012 a 0.980 ± 0.016 y la PR-AUC de 0.9995 ± 0.0003 a 0.9931 ± 0.0013. Ambas medias quedan por debajo de media_base − 2·std_base, por lo que, según la regla fijada, **la secuencia aporta** en validación. La diferencia se concentra en falsos positivos: en el umbral F2, W=1 comete 124 ± 109 contra 2.4 ± 1.3 del baseline.
- W=1 es menos estable entre semillas: su umbral F2 va de 0.010 a 0.999, y la semilla 4 deja 61 falsos negativos.
- **Aleatorio contra el baseline:** con las mismas filas en cada split y solo el orden sorteado dentro de cada minuto, el recall con umbral 0.5 baja a 0.976 ± 0.006 y la PR-AUC a 0.9966 ± 0.0014. Ambas cumplen la regla: **hay dependencia del orden del CSV**. En el umbral F2 hay 20.8 ± 10.6 falsos negativos y 117.2 ± 17.9 falsos positivos.
- Con el orden sorteado, W=10 queda al nivel de W=1: F2 de 0.979 contra 0.981, falsos positivos en el umbral F2 de 117 contra 124 y PR-AUC de 0.9966 contra 0.9931. Esta comparación no tiene regla fijada de antemano y se reporta solo como descriptiva.
- **Qué representa el orden del CSV (sección 5):** el archivo no está en orden cronológico. Cerca del 30% de las filas consecutivas retroceden más de 120 s en el tiempo, tanto por inicio como por fin del flujo. Dentro de un mismo minuto, dos filas consecutivas comparten Source IP el 68.6% de las veces (21.7% esperado al azar) y Destination IP el 47.1% (15.4% esperado). El orden agrupa los flujos por host, así que en la variante csv los flujos previos de cada ventana suelen venir del mismo host que el flujo objetivo.
- **Implicación:** en validación, la ventaja de W=10 sobre W=1 se explica por el agrupamiento por host del archivo, no por un patrón temporal entre flujos. Como Timestamp tiene resolución de minuto, ese orden no puede reconstruirse a partir de los datos. Sin él, el GRU detecta los ataques a partir de las features de cada flujo (recall ≈ 0.98 con W=1).